# prefopt: ORPO Aligned program

In [ ]:
import logging
from pathlib import Path
import os, sys
import json
from pprint import pprint

from prefopt import decode_all

## Run the trained model on dev.txt

In [ ]:
basemodel = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cpu"
inputfile = os.path.join("..", "data", "input", "dev.txt")

logging.basicConfig(filename='log.txt', filemode='w', level=logging.DEBUG)

print("Running prefopt.py on dev set...")
print("NOTE: If you trained using ORPO + LoRA,")
print("      your LoRA weights should be inside ./model/")

%%capture output
decode_all(basemodel, device, inputfile)

Path("output.txt").write_text(output.stdout)
print("Saved predictions to output.txt")

## Evaluate the dev score

In [ ]:
print("Evaluating dev score...")
!python ../output_check.py -t ../data/reference/dev.out -o output.txt

```dev score = 53.00```

**Documentation**

**1\. Overview**

The objective of Homework 4 is to convert a base Large Language Model (Qwen/Qwen2.5-0.5B-Instruct) into a better instruction-following model using **ORPO (Odds-Ratio Preference Optimization)**. ORPO is a single-model preference optimization method that eliminates the need for a reference model and instead relies on (chosen, rejected) output pairs.

Our pipeline performs:
1.  Dataset construction (prompt + constraints + chosen output + rejected output)
2.  4-bit quantized model loading
3.  ORPO + LoRA training
4.  LoRA merge and inference using `prefopt.py`
5.  Final prediction generation and evaluation

Our trained model achieved a **dev score of 53 (A+).**

**2\. Data Preprocessing & Dataset Construction**

### **2.1 Loading and Extracting Data**

The Unnatural Instructions dataset is processed as follows:
-   If an unzipped JSON file already exists, we load it directly (no repeated decompression).
-   For each entry:
    -   Extract `prompt`, `constraints`, and the reference `output` (chosen).
    -   Combine prompt + constraints into the final formatted prompt.
-   Rejected outputs are generated by running the **default baseline LLM** on each prompt.\
    These serve as the negative examples required by ORPO.

### **2.2 Dataset Shuffling & Splitting**

To ensure memory efficiency:

-   The dataset is **randomly shuffled** first.
-   We select **8000 examples** for training due to GPU memory limitations.
-   We split:
    -   **99% for training**
    -   **1% for evaluation**

This maintains dataset diversity while enabling stable training under limited resources.

**3\. Quantization Strategy**

### **3.1 Why Quantization Was Necessary**

The base Qwen2.5 0.5B model + LoRA adapters + optimizer states do not fit into our available GPU memory in full precision.

### **3.2 What We Used**

-   **4-bit quantization (NF4 / QLoRA)**

-   This reduces memory consumption enough to:
    -   Fit the full model
    -   Run ORPO
    -   Keep training stable
Without 4-bit quantization, training was not possible on the available hardware.

**4\. ORPO + LoRA Training Setup**

### **4.1 LoRA Configuration**

We followed the recommended LoRA settings:

| Parameter | Value |
| --- | --- |
| r | **16** |
| alpha | **32** |
| dropout | **0.05** |

These values provide a good trade-off between memory, stability, and alignment performance.

### **4.2 ORPO Configuration**

-   **max_steps = 1000**
    -   250 steps → too little learning
    -   500 steps → improved but still insufficient
    -   **1000 steps produced optimal dev score (53)**

-   **batch size = 1**
    -   Increasing batch size caused **constant CUDA OOM errors**
    -   Batch size 1 was the only stable configuration

-   **optimizer = rmsprop**

-   **scheduler = cosine**, which stabilizes ORPO fine-tuning

-   **gradient checkpointing = enabled**
    -   Reduces VRAM usage
    -   Prevents crashes
    -   Saves mid-training progress

### **4.3 Additional Training Notes**

-   Used **bf16** where supported
-   LoRA used to avoid committing large model files and reduce VRAM needs
-   Checkpointing ensures progress is not lost due to long runtime or GPU instability

**5\. Inference Pipeline (prefopt.py)**

`prefopt.py` performs:

1.  Load base Qwen2.5-0.5B model
2.  Load LoRA weights from `/model`
3.  Merge LoRA weights into the model
4.  Run inference using HF pipeline
5.  Generate constrained outputs
6.  Print JSON dicts formatted for `output_check.py`
This fully replaces the default baseline model and aligns with the assignment requirements.


**Analysis**

**1\. Performance Comparison**

| Model | Dev Score | Notes |
| --- | --- | --- |
| Default baseline | ~29 | Provided starter model |
| Our ORPO-trained model | **53** | A+ performance |

Our model improved by **+24 points**, the highest score bracket available.

**2\. What Worked Well**

### ORPO + LoRA Training
-   ORPO effectively uses (chosen, rejected) examples to guide the model toward proper instruction-following.
-   LoRA allows training without updating all model parameters, significantly reducing VRAM usage.

### 4-bit Quantization
-   Absolutely required for our hardware setup.
-   Allowed us to load and train the model with LoRA.

### Training for 1000 Steps
We experimented with different step counts:

| Steps | Result |
| --- | --- |
| 250 | Poor performance |
| 500 | Better but unstable |
| **1000** | **Best overall accuracy and stability** |

### Dataset Shuffling + 8k Subset

-   Reduced VRAM usage
-   Ensured input diversity
-   Prevented overfitting to early instruction templates

### Cosine Scheduler + Gradient Checkpointing

-   Cosine warmup allowed smooth loss decay
-   Checkpointing prevented OOM crashes
-   Training was restartable in case of failures

**3\. What Did NOT Work**

### Increasing Batch Size
Batch size > 1 consistently caused:
-   CUDA out-of-memory errors
-   Training halts
-   Unrecoverable crashes
Therefore, we strictly used batch size = 1.

### Shorter Training Runs
-   ORPO requires sufficient exposure to preference pairs
-   Short runs (250--500 steps) produced unstable or improperly formatted outputs

### Full Precision
-   Impossible on our hardware
-   Even loading the model caused OOM

**4\. Future Improvements**
-   Try 8-bit + 4-bit hybrid quantization
-   Increase dataset size beyond 8k samples
-   Train a reward model for further alignment (outside scope)
-   Experiment with PPO/RLHF approaches
-   Evaluate structured decoding constraints